# scRareBench | High-Level MrVI | Dataset 4 — COVID-19 autoimmunity PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Colab notebook runs **MrVI** on scRareBench Dataset 4
(`covid19_autoimmunity_pbmc`) through:

`load_dataset(4) → MethodSpec(MrVI) → benchmark_method() → scIB + rare benchmark → HTML / multi-seed bundle`

Pinned scRareBench commit:

`cfdb720f5488b9ceef9a225b35fc7558bcb0251c`

At notebook creation time, GitHub `main` is identical to that commit.

## Dataset 4 package contract

- 97,499 cells × 21,858 features
- benchmark label: `cell_type` — **evaluation only; never supplied to MrVI**
- integration/evaluation batch: `donor_id`
- canonical raw counts: `layers["counts"]`
- 22 donors and 26 label categories
- registered rare-scenario metadata is package-owned
- scIB-reference HVG policy: `evaluation_batch`

## MrVI policy for Dataset 4

- `donor_id` is the MrVI `sample_key`
- no separate nuisance `batch_key`
- method-side HVG selection: **donor-aware Seurat-v3, 4,000 HVGs, span=0.3**
- benchmark representation: **MrVI u latent** (`give_z=False`)
- biological labels are never used by the method runner

Dataset preparation, scenario attachment, scIB preprocessing, evaluation,
reporting, and multi-seed delivery remain package-owned.

Use a **GPU + High-RAM Colab**. For a first smoke run, temporarily set
`METHOD_SEEDS = [42]`.

## 1. Install the pinned package revision and MrVI runtime

In [ ]:
import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCRAREBENCH_GITHUB,
])

METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)

from scrarebench.runtime import setup_runtime, print_install_report

install_report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("scvi", "scvi.external"),
    quiet=False,
)
print_install_report(install_report)

subprocess.check_call([
    sys.executable,
    "-c",
    (
        "import scvi; "
        "from scvi.external import MRVI; "
        "print('scvi-tools', scvi.__version__); "
        "print('MRVI import OK')"
    ),
])


## 2. Load Dataset 4 through scRareBench

In [ ]:
import scrarebench
from scrarebench import dataset_info, load_dataset, scenario_table_from_adata

adata = load_dataset(4)
info = dataset_info(adata)

# Contract validation only; package-owned preparation is not duplicated.
expected = {
    "dataset_key": "covid19_autoimmunity_pbmc",
    "label_key": "cell_type",
    "batch_key": "donor_id",
    "count_layer": "counts",
    "scib_hvg_batch_mode": "evaluation_batch",
    "benchmark_ready": True,
}
for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Unexpected Dataset 4 contract for {key!r}: "
            f"observed={info.get(key)!r}, expected={value!r}"
        )

if adata.shape != (97_499, 21_858):
    raise RuntimeError(
        f"Unexpected Dataset 4 shape: {adata.shape}; expected (97499, 21858)."
    )
if "counts" not in adata.layers:
    raise RuntimeError("Dataset 4 canonical raw-count layer 'counts' is missing.")
if not adata.obs_names.is_unique:
    raise RuntimeError("Dataset 4 observation names are not unique.")
if not adata.var_names.is_unique:
    raise RuntimeError("Dataset 4 feature names are not unique.")
if adata.obs["donor_id"].astype(str).nunique() != 22:
    raise RuntimeError("Unexpected Dataset 4 donor cardinality; expected 22.")
if adata.obs["cell_type"].astype(str).nunique() != 26:
    raise RuntimeError("Unexpected Dataset 4 label cardinality; expected 26.")

scenario_table = scenario_table_from_adata(adata)
if scenario_table is None or scenario_table.empty:
    raise RuntimeError("Dataset 4 registered scenario metadata is missing.")
if len(scenario_table) != 12:
    raise RuntimeError(
        f"Unexpected Dataset 4 scenario-table size: {len(scenario_table)}; "
        "expected 12 registered six-state rows."
    )

print("scRareBench:", scrarebench.__version__)
print(adata)
print("\nDataset contract:")
for key in (
    "dataset_key", "label_key", "batch_key", "count_layer",
    "scib_hvg_batch_mode", "benchmark_ready",
):
    print(f"  {key}: {info.get(key)}")

print("Donors:", adata.obs["donor_id"].astype(str).nunique())
print("Cell-type categories:", adata.obs["cell_type"].astype(str).nunique())
print("Registered six-state rows:", len(scenario_table))
print("Scenario counts:")
print(scenario_table["scenario"].value_counts().sort_index())


## 3. Reproducibility and MrVI configuration

In [ ]:
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a first smoke run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_span": 0.3,
    "hvg_batch_key": "donor_id",

    "sample_key": "donor_id",
    "nuisance_batch_key": None,

    "n_latent_u": 30,
    "n_latent_z": 30,
    "encoder_n_hidden": 128,
    "encoder_n_layers": 2,

    "max_epochs": 300,
    "train_size": 0.90,
    "batch_size": 128,
    "early_stopping": True,
    "early_stopping_patience": 25,
}

OUTPUT_DIR = "/content/scrarebench_mrvi_dataset4_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_mrvi_dataset4_method_runs"


## 4. User-owned MrVI adapter

This is the only method-specific section.

The runner selects donor-aware 4,000 Seurat-v3 HVGs from the package-provided
canonical counts, registers `donor_id` as the MrVI `sample_key`, introduces no
unvalidated nuisance batch variable, trains MrVI, and returns the **u latent**.

The benchmark biological label is not referenced by this runner.


In [ ]:
from pathlib import Path
import gc
import random
import shutil

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
import scvi
from scvi.external import MRVI

from scrarebench import MethodOutput, dataset_info


def run_mrvi(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    dataset_batch_key = method_info["batch_key"]
    count_layer = method_info["count_layer"]

    if method_info.get("dataset_key") != "covid19_autoimmunity_pbmc":
        raise RuntimeError(
            "This runner is configured only for scRareBench Dataset 4."
        )
    if method_info.get("scib_hvg_batch_mode") != "evaluation_batch":
        raise RuntimeError(
            "Dataset 4 is expected to use evaluation-batch-aware HVGs."
        )
    if dataset_batch_key != "donor_id":
        raise RuntimeError(
            f"Unexpected Dataset 4 batch key {dataset_batch_key!r}; "
            "expected 'donor_id'."
        )
    if not count_layer or count_layer not in method_adata.layers:
        raise KeyError(
            f"Canonical raw-count layer {count_layer!r} is unavailable."
        )

    sample_key = str(config["sample_key"])
    hvg_batch_key = str(config["hvg_batch_key"])
    nuisance_batch_key = config["nuisance_batch_key"]

    if sample_key != dataset_batch_key:
        raise RuntimeError(
            f"MrVI sample_key must match the Dataset 4 donor batch: "
            f"sample_key={sample_key!r}, dataset_batch={dataset_batch_key!r}."
        )
    if hvg_batch_key != dataset_batch_key:
        raise RuntimeError(
            f"Dataset 4 method-side HVGs must be donor-aware: "
            f"hvg_batch_key={hvg_batch_key!r}, dataset_batch={dataset_batch_key!r}."
        )
    if sample_key not in method_adata.obs.columns:
        raise KeyError(f"MrVI sample key {sample_key!r} is missing.")
    if hvg_batch_key not in method_adata.obs.columns:
        raise KeyError(f"HVG batch key {hvg_batch_key!r} is missing.")
    if (
        nuisance_batch_key is not None
        and nuisance_batch_key not in method_adata.obs.columns
    ):
        raise KeyError(
            f"MrVI nuisance batch key {nuisance_batch_key!r} is missing."
        )

    original_barcodes = method_adata.obs_names.astype(str).to_numpy(copy=True)
    if len(np.unique(original_barcodes)) != len(original_barcodes):
        raise RuntimeError("Benchmark cell identifiers must be unique.")

    # Method-owned preprocessing:
    # donor-aware Seurat-v3, 4000 HVGs, span=.3.
    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        batch_key=hvg_batch_key,
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )

    if "highly_variable" not in method_adata.var.columns:
        raise RuntimeError(
            "Seurat-v3 HVG selection did not return a mask."
        )

    hvg_mask = (
        method_adata.var["highly_variable"]
        .fillna(False)
        .to_numpy(dtype=bool)
    )
    expected_hvg = min(int(config["n_hvg"]), method_adata.n_vars)
    selected_hvg = int(hvg_mask.sum())
    if selected_hvg != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} HVGs, selected {selected_hvg}."
        )

    hvg_names = method_adata.var_names[hvg_mask]
    counts_hvg = method_adata.layers[count_layer][:, hvg_mask]

    if sp.issparse(counts_hvg):
        counts_hvg = counts_hvg.tocsr().astype(np.float32)
    else:
        counts_hvg = np.asarray(counts_hvg, dtype=np.float32)

    # Count-only MrVI input. Benchmark/evaluation data remain separate.
    mrvi_adata = ad.AnnData(
        X=counts_hvg,
        obs=method_adata.obs.copy(),
        var=method_adata.var.loc[hvg_names].copy(),
    )

    if not mrvi_adata.var_names.is_unique:
        raise RuntimeError("Dataset 4 HVG identifiers are non-unique.")

    if not np.array_equal(
        mrvi_adata.obs_names.astype(str).to_numpy(),
        original_barcodes,
    ):
        raise RuntimeError(
            "Cell identity/order changed during MrVI preprocessing."
        )

    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    scvi.settings.seed = int(seed)

    # donor_id is the target sample variable.
    # No separate nuisance batch is invented.
    setup_kwargs = {
        "sample_key": sample_key,
        "backend": "torch",
    }
    if nuisance_batch_key is not None:
        setup_kwargs["batch_key"] = str(nuisance_batch_key)

    MRVI.setup_anndata(mrvi_adata, **setup_kwargs)

    model = MRVI(
        mrvi_adata,
        n_latent=int(config["n_latent_z"]),
        n_latent_u=int(config["n_latent_u"]),
        encoder_n_hidden=int(config["encoder_n_hidden"]),
        encoder_n_layers=int(config["encoder_n_layers"]),
    )

    model.train(
        max_epochs=int(config["max_epochs"]),
        accelerator="auto",
        devices="auto",
        train_size=float(config["train_size"]),
        validation_size=1.0 - float(config["train_size"]),
        batch_size=int(config["batch_size"]),
        early_stopping=bool(config["early_stopping"]),
        early_stopping_patience=int(config["early_stopping_patience"]),
        check_val_every_n_epoch=1,
        enable_progress_bar=True,
    )

    # u is the sample-invariant MrVI representation used for benchmarking.
    latent = np.asarray(
        model.get_latent_representation(
            adata=mrvi_adata,
            batch_size=int(config["batch_size"]),
            use_mean=True,
            give_z=False,
        ),
        dtype=np.float32,
    )

    expected_shape = (
        method_adata.n_obs,
        int(config["n_latent_u"]),
    )
    if latent.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected MrVI-u latent shape: {latent.shape}; "
            f"expected {expected_shape}."
        )
    if not np.isfinite(latent).all():
        raise ValueError(
            "MrVI-u latent contains NaN or infinite values."
        )

    output_barcodes = mrvi_adata.obs_names.astype(str).to_numpy(copy=True)
    if not np.array_equal(output_barcodes, original_barcodes):
        raise RuntimeError(
            "MrVI output barcode order differs from Dataset 4."
        )

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    if seed_dir.exists():
        shutil.rmtree(seed_dir)
    seed_dir.mkdir(parents=True, exist_ok=True)

    provenance = {}
    history_frames = []
    for metric, values in getattr(model, "history", {}).items():
        frame = pd.DataFrame(values).copy()
        frame["metric"] = str(metric)
        frame["epoch"] = frame.index
        history_frames.append(frame.reset_index(drop=True))

    if history_frames:
        history_df = pd.concat(history_frames, ignore_index=True)
        history_path = seed_dir / "mrvi_training_history.csv"
        history_df.to_csv(history_path, index=False)
        provenance["mrvi_training_history"] = history_path

    print(
        f"Seed {seed}: MrVI-u latent={latent.shape}, "
        f"samples={mrvi_adata.obs[sample_key].astype(str).nunique()}, "
        f"HVGs={mrvi_adata.n_vars}"
    )

    del model, mrvi_adata, counts_hvg
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=original_barcodes,
        representation_key="X_mrvi_u",
        provenance_files=provenance,
    )


## 5. Run the high-level multi-seed benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="MrVI-u",
    runner=run_mrvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        # Dataset 4 metadata supplies scib_hvg_batch_mode="evaluation_batch".
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    copy_input_per_seed=True,
)


## 6. Inspect the multi-seed result and alignment

In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Method:", result.method)
print("Completed seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  alignment:", run.alignment)
    print("  output:", run.output_dir)
    print("  interactive report:", run.interactive_report_path)
    print("  bundle:", run.bundle_path)

    if run.representation_key != "X_mrvi_u":
        raise RuntimeError(
            f"Seed {seed} returned an unexpected representation key."
        )
    if run.alignment.get("exact_order_match") is not True:
        raise RuntimeError(
            f"Seed {seed} did not preserve exact barcode order."
        )


## 7. Download the final multi-seed archive

In [ ]:
from pathlib import Path

if result.archive_path is None:
    raise RuntimeError(
        "No multi-seed archive was created. "
        "Run all final seeds with finalize=True."
    )

archive_path = Path(result.archive_path)
if not archive_path.exists():
    raise FileNotFoundError(
        f"Multi-seed archive was not found: {archive_path}"
    )

print(f"Downloading: {archive_path.name}")
print(f"Size: {archive_path.stat().st_size / (1024**2):.2f} MB")

try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Not running in Colab; archive remains at:", archive_path)


## Runtime note

For the first Colab execution, temporarily use:

```python
METHOD_SEEDS = [42]
```

After the smoke run succeeds, restore:

```python
METHOD_SEEDS = [42, 123, 2026]
```

for the final benchmark and multi-seed archive.

This notebook is **statically validated**. Runtime success is not claimed until
the actual Dataset 4 + MrVI Colab execution completes.
